In [1]:
import os

def find(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

SRC = find("10_train_student_d32.py")
s = open(SRC).read()

def rep(a, b):
    global s
    assert s.count(a) == 1, f"pola tidak ditemukan tepat satu kali: {a[:60]!r}"
    s = s.replace(a, b)

# 1) opsi baru
rep('    accum=1,', '    side_csv="",        # knee_side.csv: citra koronal & aksial lutut kiri dicerminkan\n    accum=1,')
rep('    ap.add_argument("--accum", type=int, default=CFG["accum"])',
    '    ap.add_argument("--accum", type=int, default=CFG["accum"])\n    ap.add_argument("--side_csv", default=CFG["side_csv"])')
# 2) dataset: cerminkan view koronal & aksial untuk lutut kiri (dua baris diganti terpisah)
rep('    def __init__(self, uids, ymat, vol_dir, train):',
    '    def __init__(self, uids, ymat, vol_dir, train, flip=None, flip_idx=()):')
rep('        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train',
    '        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train\n'
    '        self.flip, self.flip_idx = (flip or {}), list(flip_idx)')
rep('        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)',
    '        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)\n'
    '        if self.flip_idx and self.flip.get(self.uids[i], False):\n'
    '            vol[self.flip_idx] = vol[self.flip_idx].flip(-1)          # lutut kiri -> cermin kiri-kanan')
# 3) peta sisi lutut dan indeks view yang dicerminkan
rep('    have = set(vol_map)',
    '    have = set(vol_map)\n'
    '    FLIP, FLIP_IDX = {}, []\n'
    '    if CFG["side_csv"]:\n'
    '        sd = pd.read_csv(find_file(CFG["side_csv"]))\n'
    '        FLIP = dict(zip(sd["StudyInstanceUID"], sd["side"] == "L"))\n'
    '        u0 = next(iter(vol_map)); nv = int(np.load(os.path.join(vol_map[u0], f"{u0}.npz"))["mask"].shape[0])\n'
    '        FLIP_IDX = {3: [1, 2], 5: [1, 2, 4]}[nv]          # v1: sag, cor, ax | v2/v3: sag, cor, ax, sag, cor\n'
    '        print(f"Normalisasi sisi lutut: {sum(FLIP.get(u, False) for u in have)} studi kiri dicerminkan | view {FLIP_IDX}")')
rep('                    vol_map, train)', '                    vol_map, train, FLIP, FLIP_IDX)')

open("/kaggle/working/11_train_student_lat.py", "w").write(s)
import ast; ast.parse(s)                         # cek sintaks
print("Script dibuat dari:", SRC)
print("Cek baris baru:")
for line in s.splitlines():
    if "flip" in line.lower() and ("def __init__" in line or "self.flip" in line or "FLIP_IDX =" in line or ".flip(-1)" in line):
        print("  ", line.strip())

Script dibuat dari: /kaggle/input/notebooks/kellenvalerie/rsna-knee-14-student-d32/10_train_student_d32.py
Cek baris baru:
   def __init__(self, uids, ymat, vol_dir, train, flip=None, flip_idx=()):
   self.flip, self.flip_idx = (flip or {}), list(flip_idx)
   if self.flip_idx and self.flip.get(self.uids[i], False):
   vol[self.flip_idx] = vol[self.flip_idx].flip(-1)          # lutut kiri -> cermin kiri-kanan
   FLIP, FLIP_IDX = {}, []
   FLIP_IDX = {3: [1, 2], 5: [1, 2, 4]}[nv]          # v1: sag, cor, ax | v2/v3: sag, cor, ax, sag, cor


In [2]:
%%bash
cd /kaggle/working
COMMON="--alpha 0.5 --oof oof_effb0_kf2.csv --meta _meta.csv --backbone vit_small_patch14_dinov2.lvd142m \
        --lr_backbone 1e-4 --llrd 0.8 --batch 4 --epochs 12 --side_csv knee_side.csv"

CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $COMMON --seed 42 --tag dinov2s_student_v1d_lat \
    > log_dinov2s_student_v1d_lat.txt 2>&1 &
CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $COMMON --seed 7 --tag dinov2s_student_v1d_lat_s7 \
    > log_dinov2s_student_v1d_lat_s7.txt 2>&1 &
wait
for t in dinov2s_student_v1d_lat dinov2s_student_v1d_lat_s7; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Normalisasi\|Train \|Label  :\|Error\|error" log_*.txt

===== dinov2s_student_v1d_lat =====
Epoch 11: loss 0.3745 | val AUC 0.8820 | val-ref AUC 0.8964 | gold AUC 0.8631 | 13.2 menit

=== [dinov2s_student_v1d_lat] Checkpoint terbaik (val-ref AUC 0.8964) pada label emas ===
ACL                AUC=0.908
MCL                AUC=0.930
Medial Meniscus    AUC=0.880
Lateral Meniscus   AUC=0.727
Medial OA          AUC=0.955
Lateral OA         AUC=0.816
PF OA              AUC=0.812
Effusion           AUC=0.888
Synovitis          AUC=0.714
Baker's            AUC=0.962
Contusion          AUC=0.883
Fracture           AUC=0.882
Mean AUC gold: 0.8631
===== dinov2s_student_v1d_lat_s7 =====
Epoch 11: loss 0.3722 | val AUC 0.8780 | val-ref AUC 0.8940 | gold AUC 0.8624 | 13.5 menit

=== [dinov2s_student_v1d_lat_s7] Checkpoint terbaik (val-ref AUC 0.8940) pada label emas ===
ACL                AUC=0.902
MCL                AUC=0.916
Medial Meniscus    AUC=0.868
Lateral Meniscus   AUC=0.771
Medial OA          AUC=0.969
Lateral OA         AUC=0.795
PF OA         

In [3]:
import pandas as pd
for tag in ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7"]:
    print(f"\n===== {tag} =====")
    print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))


===== dinov2s_student_v1d_lat =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5427   0.7521       0.7658    0.7743 13.2378
     1 0.4707   0.8154       0.8333    0.8261 13.1405
     2 0.4470   0.8361       0.8535    0.8293 13.1765
     3 0.4353   0.8530       0.8698    0.8501 13.1733
     4 0.4236   0.8547       0.8702    0.8518 13.1796
     5 0.4137   0.8689       0.8821    0.8549 13.1495
     6 0.4059   0.8733       0.8901    0.8583 13.1119
     7 0.3961   0.8764       0.8913    0.8696 13.1450
     8 0.3882   0.8775       0.8927    0.8701 13.1575
     9 0.3812   0.8793       0.8943    0.8620 13.2036
    10 0.3769   0.8819       0.8960    0.8609 13.2057
    11 0.3745   0.8820       0.8964    0.8631 13.1766

===== dinov2s_student_v1d_lat_s7 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5299   0.8048       0.8196    0.8246 13.6493
     1 0.4607   0.8387       0.8532    0.8510 13.5513
     2 0.4412   0.8430       0.8587    0.8471 13.5587
     